# N-gram Frequency Analysis for Engram-Lite

Analyze bigram/trigram distributions in fineweb-edu to inform hash table sizing.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Configuration
NUM_DOCS = 10000
VOCAB_SIZE = 50304
TABLE_MULTIPLIERS = [1, 2, 5, 10]
HASH_PRIMES = (36313, 27191)  # (curr_mult, prev_mult)

In [ ]:
# Load tokenizer and stream dataset
tokenizer = AutoTokenizer.from_pretrained("gpt2")
dataset = load_dataset("karpathy/fineweb-edu-100b-shuffle", split="train", streaming=True)

BOS_TOKEN = tokenizer.bos_token_id or tokenizer.eos_token_id  # GPT-2 uses eos as bos

# Tokenize documents, inserting BOS between docs to avoid cross-document bigrams
all_tokens = []
for i, example in enumerate(dataset):
    if i >= NUM_DOCS:
        break
    if all_tokens:  # Insert separator before each doc (except first)
        all_tokens.append(BOS_TOKEN)
    tokens = tokenizer.encode(example["text"], add_special_tokens=False)
    all_tokens.extend(tokens)
    if (i + 1) % 10000 == 0:
        print(f"Processed {i + 1} documents, {len(all_tokens):,} tokens")

all_tokens = np.array(all_tokens, dtype=np.int32)
print(f"\nTotal: {len(all_tokens):,} tokens from {NUM_DOCS:,} documents")
print(f"BOS token id: {BOS_TOKEN} ({repr(tokenizer.decode([BOS_TOKEN]))})")

## Raw Bigram Distribution (Before Hashing)

In [ ]:
# Extract bigrams as (prev, curr) tuples, excluding BOS boundaries
prev_tokens = all_tokens[:-1]
curr_tokens = all_tokens[1:]

# Filter out bigrams involving BOS token
valid_mask = (prev_tokens != BOS_TOKEN) & (curr_tokens != BOS_TOKEN)
prev_tokens = prev_tokens[valid_mask]
curr_tokens = curr_tokens[valid_mask]
bigrams = list(zip(prev_tokens.tolist(), curr_tokens.tolist()))

# Count bigram frequencies
bigram_counts = Counter(bigrams)
unique_bigrams = len(bigram_counts)
total_bigrams = len(bigrams)
theoretical_max = VOCAB_SIZE ** 2

print(f"Unique bigrams observed: {unique_bigrams:,}")
print(f"Total bigram occurrences: {total_bigrams:,}")
print(f"Theoretical max (V^2): {theoretical_max:,}")
print(f"Coverage: {100 * unique_bigrams / theoretical_max:.4f}%")
print(f"(Excluded {(~valid_mask).sum():,} bigrams at document boundaries)")

In [ ]:
# Frequency distribution: how many bigrams appear N times
freq_counts = Counter(bigram_counts.values())
freq_items = sorted(freq_counts.items())

print("Frequency distribution (count -> num_bigrams):")
print(f"  Appear 1x:    {freq_counts.get(1, 0):,} bigrams")
print(f"  Appear 2x:    {freq_counts.get(2, 0):,} bigrams")
print(f"  Appear 3-10x: {sum(freq_counts.get(i, 0) for i in range(3, 11)):,} bigrams")
print(f"  Appear 10-100x: {sum(freq_counts.get(i, 0) for i in range(10, 101)):,} bigrams")
print(f"  Appear 100+x: {sum(c for f, c in freq_counts.items() if f >= 100):,} bigrams")

In [ ]:
# Log-log rank-frequency plot (Zipf's law)
frequencies = sorted(bigram_counts.values(), reverse=True)
ranks = np.arange(1, len(frequencies) + 1)

plt.figure(figsize=(10, 6))
plt.loglog(ranks, frequencies, 'b-', alpha=0.7, linewidth=0.5)
plt.xlabel('Rank')
plt.ylabel('Frequency')
plt.title('Bigram Rank-Frequency Distribution (Zipf\'s Law)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Top-20 most common bigrams (decoded to text)
print("Top 20 most common bigrams:")
for (prev_id, curr_id), count in bigram_counts.most_common(20):
    prev_tok = tokenizer.decode([prev_id])
    curr_tok = tokenizer.decode([curr_id])
    print(f"  {repr(prev_tok):>15} -> {repr(curr_tok):<15}: {count:>8,} ({100*count/total_bigrams:.2f}%)")

In [ ]:
# Cumulative coverage: top-N bigrams cover what % of occurrences?
cumsum = np.cumsum(frequencies)
coverage_pct = 100 * cumsum / total_bigrams

milestones = [50, 80, 90, 95, 99]
print("Cumulative coverage:")
for m in milestones:
    idx = np.searchsorted(coverage_pct, m)
    print(f"  Top {idx:,} bigrams cover {m}% of occurrences")

plt.figure(figsize=(10, 6))
plt.semilogx(ranks, coverage_pct)
plt.xlabel('Number of top bigrams (log scale)')
plt.ylabel('Cumulative coverage (%)')
plt.title('Cumulative Bigram Coverage')
plt.axhline(y=90, color='r', linestyle='--', alpha=0.5, label='90%')
plt.axhline(y=99, color='g', linestyle='--', alpha=0.5, label='99%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Hash Collision Analysis

In [ ]:
def bigram_hash(prev_id, curr_id, table_size):
    """Match engram_lite: (36313*curr) ^ (27191*prev) % (table_size - 1)"""
    return ((HASH_PRIMES[0] * curr_id) ^ (HASH_PRIMES[1] * prev_id)) % (table_size - 1)

def analyze_collisions(bigram_counts, table_multiplier):
    """Analyze hash collisions for a given table size."""
    table_size = VOCAB_SIZE * table_multiplier
    
    # Map each unique bigram to its hash
    hash_to_bigrams = {}
    for (prev_id, curr_id), count in bigram_counts.items():
        h = bigram_hash(prev_id, curr_id, table_size)
        if h not in hash_to_bigrams:
            hash_to_bigrams[h] = []
        hash_to_bigrams[h].append(((prev_id, curr_id), count))
    
    unique_hashes = len(hash_to_bigrams)
    unique_bigrams = len(bigram_counts)
    collision_rate = 1 - (unique_hashes / unique_bigrams)
    
    # Weighted collision rate (by frequency)
    total_weighted = sum(bigram_counts.values())
    colliding_weighted = 0
    for h, items in hash_to_bigrams.items():
        if len(items) > 1:
            colliding_weighted += sum(count for _, count in items)
    weighted_collision_rate = colliding_weighted / total_weighted
    
    # Collision distribution
    collision_sizes = [len(items) for items in hash_to_bigrams.values()]
    
    return {
        "table_size": table_size,
        "unique_hashes": unique_hashes,
        "unique_bigrams": unique_bigrams,
        "collision_rate": collision_rate,
        "weighted_collision_rate": weighted_collision_rate,
        "collision_sizes": collision_sizes,
        "max_collision": max(collision_sizes),
    }

In [ ]:
# Analyze collisions for each table multiplier
results = []
for mult in TABLE_MULTIPLIERS:
    result = analyze_collisions(bigram_counts, mult)
    results.append(result)
    print(f"\nTable multiplier: {mult} (table_size = {result['table_size']:,})")
    print(f"  Unique hashes: {result['unique_hashes']:,} / {result['unique_bigrams']:,} bigrams")
    print(f"  Collision rate: {100*result['collision_rate']:.2f}%")
    print(f"  Weighted collision rate: {100*result['weighted_collision_rate']:.2f}%")
    print(f"  Max collision (bigrams per bucket): {result['max_collision']}")

In [ ]:
# Plot: collision rate vs table_multiplier
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

mults = TABLE_MULTIPLIERS
collision_rates = [100 * r["collision_rate"] for r in results]
weighted_rates = [100 * r["weighted_collision_rate"] for r in results]

ax1.plot(mults, collision_rates, 'bo-', label='Unweighted')
ax1.plot(mults, weighted_rates, 'ro-', label='Weighted by freq')
ax1.set_xlabel('Table Multiplier')
ax1.set_ylabel('Collision Rate (%)')
ax1.set_title('Collision Rate vs Table Size')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Memory cost (table_size * hidden_dim * 4 bytes for float32)
HIDDEN_DIM = 2048
memory_gb = [r["table_size"] * HIDDEN_DIM * 4 / (1024**3) for r in results]
ax2.bar(range(len(mults)), memory_gb, tick_label=[f"{m}x" for m in mults])
ax2.set_xlabel('Table Multiplier')
ax2.set_ylabel('Memory (GB)')
ax2.set_title('Embedding Table Memory Cost')
for i, (m, mem) in enumerate(zip(mults, memory_gb)):
    ax2.annotate(f'{mem:.1f} GB', (i, mem), ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Histogram: how many bigrams share each hash bucket (for table_multiplier=5)
result_5x = results[TABLE_MULTIPLIERS.index(5)]
collision_sizes = result_5x["collision_sizes"]

plt.figure(figsize=(10, 6))
max_size = min(max(collision_sizes), 20)
bins = np.arange(1, max_size + 2) - 0.5
plt.hist(collision_sizes, bins=bins, edgecolor='black', alpha=0.7)
plt.xlabel('Bigrams per hash bucket')
plt.ylabel('Number of buckets')
plt.title(f'Hash Bucket Collision Distribution (table_multiplier=5)')
plt.xticks(range(1, max_size + 1))
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

size_counts = Counter(collision_sizes)
print("Bucket distribution:")
for size in sorted(size_counts.keys())[:10]:
    print(f"  {size} bigram(s) per bucket: {size_counts[size]:,} buckets")

## Table Size Recommendations

In [ ]:
print("Summary Table:")
print(f"{'Mult':>6} {'Table Size':>12} {'Unique Hashes':>15} {'Collision':>12} {'Weighted':>12} {'Memory':>10}")
print("-" * 70)
for mult, result in zip(TABLE_MULTIPLIERS, results):
    mem_gb = result["table_size"] * HIDDEN_DIM * 4 / (1024**3)
    print(f"{mult:>6}x {result['table_size']:>12,} {result['unique_hashes']:>15,} "
          f"{100*result['collision_rate']:>11.2f}% {100*result['weighted_collision_rate']:>11.2f}% "
          f"{mem_gb:>9.1f}GB")

print(f"\nUnique bigrams observed: {unique_bigrams:,}")
print(f"\nRecommendation:")
print(f"  - table_multiplier=5 provides good collision/memory tradeoff")
print(f"  - Covers {unique_bigrams:,} unique bigrams with table_size={VOCAB_SIZE*5:,}")

## Trigram Extension (Bonus)

In [ ]:
# Extract trigrams as (prev2, prev1, curr) tuples, excluding BOS boundaries
prev2_tokens = all_tokens[:-2]
prev1_tokens = all_tokens[1:-1]
curr_tokens_tri = all_tokens[2:]

# Filter out trigrams involving BOS token
valid_mask_tri = (prev2_tokens != BOS_TOKEN) & (prev1_tokens != BOS_TOKEN) & (curr_tokens_tri != BOS_TOKEN)
prev2_tokens = prev2_tokens[valid_mask_tri]
prev1_tokens = prev1_tokens[valid_mask_tri]
curr_tokens_tri = curr_tokens_tri[valid_mask_tri]
trigrams = list(zip(prev2_tokens.tolist(), prev1_tokens.tolist(), curr_tokens_tri.tolist()))

trigram_counts = Counter(trigrams)
unique_trigrams = len(trigram_counts)
total_trigrams = len(trigrams)

print(f"Unique trigrams observed: {unique_trigrams:,}")
print(f"Total trigram occurrences: {total_trigrams:,}")
print(f"Theoretical max (V^3): {VOCAB_SIZE**3:,.0e}")
print(f"\nRatio vs bigrams: {unique_trigrams/unique_bigrams:.2f}x more unique trigrams")
print(f"(Excluded {(~valid_mask_tri).sum():,} trigrams at document boundaries)")

In [ ]:
# Trigram frequency distribution
tri_freq_counts = Counter(trigram_counts.values())

print("Trigram frequency distribution:")
print(f"  Appear 1x:    {tri_freq_counts.get(1, 0):,} trigrams ({100*tri_freq_counts.get(1,0)/unique_trigrams:.1f}%)")
print(f"  Appear 2x:    {tri_freq_counts.get(2, 0):,} trigrams")
print(f"  Appear 3-10x: {sum(tri_freq_counts.get(i, 0) for i in range(3, 11)):,} trigrams")
print(f"  Appear 10+x:  {sum(c for f, c in tri_freq_counts.items() if f >= 10):,} trigrams")

In [ ]:
# Trigram hash function (extended)
def trigram_hash(prev2_id, prev1_id, curr_id, table_size):
    """Extended hash: combine three tokens."""
    h = (36313 * curr_id) ^ (27191 * prev1_id) ^ (15373 * prev2_id)
    return h % (table_size - 1)

def analyze_trigram_collisions(trigram_counts, table_multiplier):
    """Analyze hash collisions for trigrams."""
    table_size = VOCAB_SIZE * table_multiplier
    
    hash_to_trigrams = {}
    for (prev2_id, prev1_id, curr_id), count in trigram_counts.items():
        h = trigram_hash(prev2_id, prev1_id, curr_id, table_size)
        if h not in hash_to_trigrams:
            hash_to_trigrams[h] = []
        hash_to_trigrams[h].append(count)
    
    unique_hashes = len(hash_to_trigrams)
    unique_trigrams = len(trigram_counts)
    collision_rate = 1 - (unique_hashes / unique_trigrams)
    
    return {
        "table_size": table_size,
        "unique_hashes": unique_hashes,
        "unique_trigrams": unique_trigrams,
        "collision_rate": collision_rate,
    }

print("Trigram collision analysis:")
for mult in TABLE_MULTIPLIERS:
    result = analyze_trigram_collisions(trigram_counts, mult)
    print(f"  {mult}x: {result['unique_hashes']:,} unique hashes, "
          f"{100*result['collision_rate']:.2f}% collision rate")

In [ ]:
# Compare bigram vs trigram statistics
print("Bigram vs Trigram Comparison:")
print(f"{'':>20} {'Bigram':>15} {'Trigram':>15}")
print("-" * 52)
print(f"{'Unique n-grams':>20} {unique_bigrams:>15,} {unique_trigrams:>15,}")
print(f"{'Singletons (1x)':>20} {freq_counts.get(1, 0):>15,} {tri_freq_counts.get(1, 0):>15,}")
print(f"{'Singleton %':>20} {100*freq_counts.get(1,0)/unique_bigrams:>14.1f}% {100*tri_freq_counts.get(1,0)/unique_trigrams:>14.1f}%")